# Dynamical decoupling with the Qubex provider

Idle windows in a scheduled circuit are where qubit coherence silently
decays. This notebook builds a circuit with a long idle window, then uses
`build_dynamical_decoupling_pass_manager` to fill that window with a
decoupling sequence — and shows at the pulse level that the inserted gates
become calibrated Qubex pulses, correctly spaced, without changing the
rest of the schedule.

The setup (offline `qubex.Experiment` from [qubex-config/](qubex-config/),
topology generated from the calibration data) is the same as in
[tutorial.ipynb](tutorial.ipynb) — see that notebook for the walkthrough.

Requirements:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
from qiskit import QuantumCircuit, transpile
from qubex import Experiment

from qiskit_qubex_provider import (
    QubexProvider,
    build_device_topology,
    build_dynamical_decoupling_pass_manager,
    build_pulse_schedule_timeline_figure,
    extract_pulse_timeline,
)

In [2]:
topology = build_device_topology(
    calib_note_path="qubex-config/calibration/calib_note.json",
    params_dir="qubex-config/params",
    name="4Q-DEMO",
    device_id="4Q-DEMO",
)
exp = Experiment(
    system_id="4Q-DEMO-SYS",
    muxes=[0],
    config_dir="qubex-config/config",
    params_dir="qubex-config/params",
    calib_note_path="qubex-config/calibration/calib_note.json",
    calibration_valid_days=10000,
    mock_mode=True,
)
backend = QubexProvider.from_experiment(exp, device_topology=topology).get_backend()
exp.qubit_labels

[qubex.experiment.experiment_context] WARNING: Skew file not found: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config/skew.yaml
date: 2026-06-12 20:34:01
python: 3.13.11
qubex: 1.5.0b4
env: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/.venv
config: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config
params: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/params
chip: 16Q-DEMO (16-qubit demo chip)
qubits: ['Q00', 'Q01', 'Q02', 'Q03']
muxes: ['MUX0']
boxes: ['DEMO2', 'DEMO1']


['Q00', 'Q01', 'Q02', 'Q03']

## A circuit with an idle window

`q0` runs an `x` at the start and another `x` after the barrier, while
`q2`–`q3` run a slow echoed cross-resonance CX in between. That sandwiches
a ≈660 ns idle window on `q0` — exactly where you would want dynamical
decoupling.

In [3]:
qc = QuantumCircuit(4, 4)
qc.x(0)
qc.h(3)
qc.cx(3, 2)
qc.barrier()
qc.x(0)
qc.measure(range(4), range(4))
qc.draw()

┌───┐      ░ ┌───┐      ┌─┐
q_0: ┤ X ├──────░─┤ X ├──────┤M├
     └───┘      ░ └┬─┬┘      └╥┘
q_1: ───────────░──┤M├────────╫─
          ┌───┐ ░  └╥┘ ┌─┐    ║ 
q_2: ─────┤ X ├─░───╫──┤M├────╫─
     ┌───┐└─┬─┘ ░   ║  └╥┘┌─┐ ║ 
q_3: ┤ H ├──■───░───╫───╫─┤M├─╫─
     └───┘      ░   ║   ║ └╥┘ ║ 
c: 4/═══════════════╩═══╩══╩══╩═
                    1   2  3  0

### Baseline: the idle window, visualized

Without DD, `Q00` plays its first `Drag` pulse at *t* = 0 and then sits
idle until ≈688 ns while the CX branch runs:

In [4]:
baseline = transpile(qc, backend, scheduling_method="asap", optimization_level=1)
schedule_baseline = backend.validate(baseline)[0]
build_pulse_schedule_timeline_figure(schedule_baseline, title="No DD").show()

## Insert an XY4 sequence

Build the DD pass manager from the backend — it uses the target's
calibrated gate durations to size the sequence — and run it on the routed
(but not yet scheduled) circuit. Built-in sequences are `"xx"`, `"xy4"`,
and `"x"`/`"hahn"`; knobs like `qubits` and `spacing` are forwarded to
Qiskit's `PadDynamicalDecoupling`.

In [5]:
physical = transpile(qc, backend, optimization_level=1)

dd_xy4 = build_dynamical_decoupling_pass_manager(
    backend,
    sequence="xy4",
    scheduling_method="asap",
)
circuit_xy4 = dd_xy4.run(physical)
print(dict(circuit_xy4.count_ops()))

{'delay': 10, 'x': 4, 'measure': 4, 'y': 2, 'h': 1, 'cx': 1, 'barrier': 1}


Two `x` and two `y` gates were inserted (the original circuit had two
`x`). Compile to pulses and look at the same window:

In [6]:
schedule_xy4 = backend.validate(circuit_xy4)[0]
build_pulse_schedule_timeline_figure(schedule_xy4, title="XY4 DD").show()

The idle window on `Q00` now contains four evenly spaced calibrated `Drag`
pulses (X–Y–X–Y; the Y pulses are the same waveform with a π/2 frame
shift). Everything else — the CX branch, the readouts — is untouched.
Numerically:

In [7]:
extract_pulse_timeline(schedule_xy4)["Q00"]

[{'kind': 'pulse', 'name': 'Drag', 'start_ns': 0.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 94.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 260.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 428.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 594.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 688.0, 'duration_ns': 24.0}]

## A sparser alternative: XX

The `"xx"` sequence places just two echo pulses in the same window:

In [8]:
dd_xx = build_dynamical_decoupling_pass_manager(
    backend,
    sequence="xx",
    scheduling_method="asap",
)
schedule_xx = backend.validate(dd_xx.run(physical))[0]
build_pulse_schedule_timeline_figure(schedule_xx, title="XX DD").show()

## Wrap-up

The DD-padded circuit validates and runs like any other circuit — inserted
gates become calibrated Qubex pulses and the remaining idle time becomes
`Blank` padding. For coupling-context-aware X-sequences there is also
`build_topology_aware_dynamical_decoupling_pass_manager` (wrapping
Qiskit's `ContextAwareDynamicalDecoupling`); see
[docs/dynamical-decoupling.md](../docs/dynamical-decoupling.md) for the
details and knobs, and [tutorial.ipynb](tutorial.ipynb) for the full
pipeline walkthrough.